# <h1 align="center">ADY201 — Final Project</h1>
# <h2 align="center">Phân tích hành vi mua sắm và Dự đoán đặt lại đơn hàng trên nền tảng Instacart</h2>

<div align="center">

**Bộ dữ liệu:** Instacart Market Basket Analysis (Kaggle)  
**Công cụ:** Python &nbsp;|&nbsp; SQL &nbsp;|&nbsp; R &nbsp;|&nbsp; Power BI

</div>

---
##  Mục lục

1. [Giới thiệu và mục tiêu dự án](#section-1)
2. [Cấu trúc dữ liệu](#section-2)
3. **Python Data Processing — Làm sạch và chuẩn bị dữ liệu**
   - [I. Đọc dữ liệu và kiểm tra tổng quan](#section-I)
   - [II. Xử lý giá trị thiếu (Missing Values)](#section-II)
   - [III. Kiểm tra và loại bỏ trùng lặp](#section-III)
   - [IV. Chuẩn hóa kiểu dữ liệu](#section-IV)
   - [V. Nạp dữ liệu vào SQL Server](#section-V)
4. Python EDA — Khám phá và trực quan hóa dữ liệu *(notebook tiếp theo)*
5. Python ML — Xây dựng mô hình học máy *(notebook tiếp theo)*

---
## 1. Giới thiệu và mục tiêu dự án <a id="section-1"></a>

### 1.1. Bối cảnh

**Instacart** là nền tảng giao hàng tạp hóa trực tuyến tại Mỹ, xử lý hàng triệu đơn hàng mỗi ngày. Trong môi trường cạnh tranh cao, việc hiểu hành vi người dùng "*họ mua gì, mua khi nào, các sản phẩm nào hay đi kèm nhau*" là yếu tố then chốt để:
- Tối ưu kho hàng và bố trí sản phẩm
- Cá nhân hóa gợi ý sản phẩm cho từng khách hàng
- Tăng tỷ lệ giữ chân khách hàng (Customer Retention)

Dự án khai thác bộ dữ liệu lịch sử đặt hàng thực tế của Instacart để giải quyết **4 bài toán phân tích** cụ thể.

### 1.2. Các bài toán phân tích

| Bài toán | Loại phân tích | Công cụ | Đầu ra |
|---|---|---|---|
| Phân tích hành vi mua sắm theo thời gian, tần suất và danh mục | Mô tả (Descriptive) | Python + SQL | Biểu đồ xu hướng |
| Phân khúc khách hàng theo chỉ số RFM | Phân cụm (K-Means) | SQL + Python | Nhóm khách hàng |
| Tìm sản phẩm thường được mua cùng nhau | Khai phá tập phổ biến (Apriori) | R | Luật kết hợp |
| Dự đoán khả năng khách đặt lại sản phẩm | Phân loại nhị phân | Python (XGBoost) | Xác suất đặt lại |

### 1.3. Sơ đồ luồng dữ liệu tổng thể

```
6 file CSV thô (Kaggle)
        │
        ▼
  [PYTHON — Data Processing]
  Làm sạch dữ liệu
        │
        ▼
     [SQL SERVER]
  Nạp dữ liệu, Feature Engineering, VIEW
        │
   ┌────┴──────────────┐
   ▼                   ▼
  [R]            [PYTHON EDA & ML]
  Apriori        EDA, K-Means, XGBoost
  Giỏ hàng
   │                   │
   └───────────────────┘
            │
            ▼
      [POWER BI]
      Dashboard tổng hợp
```

---
## 2. Cấu trúc dữ liệu <a id="section-2"></a>

**Nguồn:** [Kaggle — Instacart Market Basket Analysis](https://www.kaggle.com/datasets/psparks/instacart-market-basket-analysis)

Bộ dữ liệu gồm **6 file CSV** liên kết với nhau qua các khóa chung:

| File CSV | Số dòng (ước tính) | Nội dung chính |
|---|---|---|
| `orders.csv` | ~3.4 triệu | Thông tin đơn hàng: user_id, thời điểm đặt, ngày trong tuần, giờ, khoảng cách với đơn trước |
| `order_products__prior.csv` | ~32 triệu | Lịch sử sản phẩm trong các đơn cũ: product_id, thứ tự thêm vào giỏ, có đặt lại không |
| `order_products__train.csv` | ~1.4 triệu | Tập huấn luyện: sản phẩm trong đơn và nhãn `reordered` |
| `products.csv` | ~50,000 | Tên sản phẩm, danh mục và nhóm hàng |
| `aisles.csv` | 134 | Tên danh mục sản phẩm |
| `departments.csv` | 21 | Tên nhóm hàng |

**Sơ đồ quan hệ giữa các bảng:**

```
orders ──────────────────────────────────────────┐
  │ (order_id)                                    │
  ├──► order_products__prior (order_id, product_id)
  └──► order_products__train (order_id, product_id)
                                   │ (product_id)
                              products
                                   │ (aisle_id)       │ (department_id)
                                aisles          departments
```

---
# PYTHON DATA PROCESSING — LÀM SẠCH VÀ CHUẨN BỊ DỮ LIỆU

**Công cụ:** Python (Pandas, SQLAlchemy)  
**Mục tiêu:** Đọc 6 file CSV thô, phát hiện và xử lý lỗi dữ liệu, kết hợp các bảng, sau đó xuất ra các định dạng phù hợp cho các bước phân tích tiếp theo.

**Đầu ra của Python Data Processing:**

Toàn bộ 6 bảng đã làm sạch được nạp trực tiếp vào SQL Server (database `Project_ADY201m`) — không xuất file trung gian nào. Các bước tiếp theo (feature engineering, xuất `transactions.csv` cho R, Python EDA, Python ML) đều đọc thẳng từ SQL Server.


### Cài đặt và import thư viện

Các thư viện được sử dụng trong Python Data Processing:

| Thư viện | Mục đích |
|---|---|
| `pandas` | Đọc, xử lý và thao tác dữ liệu dạng bảng |
| `numpy` | Hỗ trợ tính toán số học |
| `os` | Tạo thư mục đầu ra |
| `sqlalchemy` | Kết nối và nạp dữ liệu vào SQL Server |


In [7]:
# import các thư viện

import pandas as pd

import numpy as np

import os 

import sqlalchemy

---
## I. Đọc dữ liệu và kiểm tra tổng quan <a id="section-I"></a>

**Mục tiêu:** Nạp toàn bộ 6 file CSV vào bộ nhớ và thực hiện kiểm tra ban đầu để nắm được:
- Kích thước mỗi bảng (số dòng × số cột)
- Kiểu dữ liệu của từng cột
- Số lượng giá trị bị thiếu (null/NaN) ở mỗi cột
- Vài dòng đầu tiên để xác nhận dữ liệu đọc đúng định dạng

**Kỹ thuật sử dụng:** `pd.read_csv()`, `df.shape`, `df.dtypes`, `df.isnull().sum()`, `df.head()`


In [8]:
# Khởi tạo biến chứa dữ liệu và nạp dữ liệu vào thông qua pd.read_csv()

DATA_DIR = os.path.join('..', 'Data', 'raw')

df_o   = pd.read_csv(os.path.join(DATA_DIR, 'orders.csv'))
df_opp = pd.read_csv(os.path.join(DATA_DIR, 'order_products__prior.csv'))
df_opt = pd.read_csv(os.path.join(DATA_DIR, 'order_products__train.csv'))
df_pr  = pd.read_csv(os.path.join(DATA_DIR, 'products.csv'))
df_a   = pd.read_csv(os.path.join(DATA_DIR, 'aisles.csv'))
df_de  = pd.read_csv(os.path.join(DATA_DIR, 'departments.csv'))


---





In [9]:
# Khởi tạo list bao gồm các tupble chưa tên bảng và biến data_fame tương ứng

df_all=[('order',df_o),
        ('order_products__prior',df_opp),
        ('order_products__train',df_opt),
        ('products',df_pr),
        ('aisles',df_a),
        ('departments',df_de)]

# Lặp để kiểm tra từng bảng
 
for name,data_fame in df_all:
    print(f"\n=== {name} ===")
    print(data_fame.shape)
    print(data_fame.isnull().sum())
    print(data_fame.head(3))


=== order ===
(3421083, 7)
order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64
   order_id  user_id eval_set  order_number  order_dow  order_hour_of_day  \
0   2539329        1    prior             1          2                  8   
1   2398795        1    prior             2          3                  7   
2    473747        1    prior             3          3                 12   

   days_since_prior_order  
0                     NaN  
1                    15.0  
2                    21.0  

=== order_products__prior ===
(32434489, 4)
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64
   order_id  product_id  add_to_cart_order  reordered
0         2       33120                  1          1
1         2       28985                  2       

---



Sau khi đọc dữ liệu, kiểm tra nhanh từng bảng để phát hiện vấn đề:


**Nhận xét sau kiểm tra:**  

 | Bảng | Số dòng | Số cột | Missing Values |
 |---|---|---|---|
 | `orders` | 3,421,083 | 7 | 206,209 (cột `days_since_prior_order`) |
 | `order_products__prior` | 32,434,489 | 4 | Không có |
 | `order_products__train` | 1,384,617 | 4 | Không có |
 | `products` | 49,688 | 4 | Không có |
 | `aisles` | 134 | 2 | Không có |
 | `departments` | 21 | 2 | Không có |

 **Phát hiện chính:**

 - **`orders`:** Cột `days_since_prior_order` có **206,209 giá trị null** bằng số lượng người dùng duy nhất trong tập dữ liệu. Điều này xác nhận mỗi người dùng có đúng một đơn hàng đầu tiên chưa có đơn trước để tính khoảng cách. Đây là giá trị thiếu **có chủ đích**, không phải lỗi dữ liệu nên sẽ xử lý ở **bước II**.

 - **`order_products__prior`:** Bảng lớn nhất (~32 triệu dòng), không có missing values. Đây là nền tảng chính cho toàn bộ phân tích EDA và xây dựng đặc trưng ở các bước sau.

 - **`order_products__train`:** Không có missing values. Cột `reordered` trong bảng này đóng vai trò **biến mục tiêu (target variable)** cho mô hình học máy ở phần Python ML.

 - **`aisles` và `departments`:** Hai bảng nhỏ đóng vai trò **bảng tra cứu (lookup table)**, không có vấn đề gì.

 >**Vấn đề cần xử lý ở các bước tiếp theo:**
 >1. Missing values ở `days_since_prior_order` ở **Bước II**
 >2. Kiểu dữ liệu `order_dow`, `order_hour_of_day`, `reordered` cần chuẩn hóa ở **Bước IV**

---
## II. Xử lý giá trị thiếu (Missing Values) <a id="section-II"></a>

**Vấn đề phát hiện:** Cột `days_since_prior_order` trong bảng `orders` có giá trị `NaN` ở tất cả các đơn hàng **đầu tiên** của mỗi người dùng. Điều này hoàn toàn hợp lý về mặt logic vì chưa có đơn trước đó để tính khoảng cách.

**Cách xử lý:**
1. Tạo cột mới `is_first_order` để đánh dấu đây là đơn hàng đầu tiên (giá trị `1`) hay không (giá trị `0`), giúp phân tích sau này phân biệt được hai nhóm này.
2. Điền giá trị `0` vào các ô `NaN` trong cột `days_since_prior_order`. Nhóm chọn `0` thay vì các giá trị đặc biệt khác vì model chính ở phần Python ML là XGBoost, một tree based model không bị ảnh hưởng bởi giá trị số học của missing value, và cột `is_first_order` đã đảm nhận vai trò đánh dấu nhóm đặc biệt này.
3. Xác nhận không còn giá trị thiếu sau khi xử lý.

**Kỹ thuật sử dụng:** `df.isnull()`, `df.fillna()`, `.astype(int)`


In [10]:
# Tạo cột is_first_order để đánh dấu các đơn hàng đầu tiên (1 là đơn hàng đầu, 0 là ko phải đơn hàng đầu)
df_o['is_first_order']=df_o['days_since_prior_order'].isnull().astype(int)
# Chuyển các giá trị NaN thành 0
df_o['days_since_prior_order']=df_o['days_since_prior_order'].fillna(0)

In [11]:
# Kiểm tra lại 2 cột mới khởi tạo
df_o[['days_since_prior_order','is_first_order']].head(15)

,days_since_prior_order,is_first_order
0,0.0,1
1,15.0,0
2,21.0,0
3,29.0,0
4,28.0,0
5,19.0,0
6,20.0,0
7,14.0,0
8,0.0,0
9,30.0,0


In [12]:
# Kiểm tra lại các missvalue của bảng orders
df_o.isnull().sum()

order_id                  0
user_id                   0
eval_set                  0
order_number              0
order_dow                 0
order_hour_of_day         0
days_since_prior_order    0
is_first_order            0
dtype: int64

---

> **Nhận xét:**
>
> Sau khi xử lý, cột `days_since_prior_order` không còn giá trị null. Tổng cộng có **206,209 đơn hàng đầu tiên** được đánh dấu bằng cột `is_first_order = 1`, bằng đúng số lượng người dùng duy nhất trong tập dữ liệu.

---
## III. Kiểm tra và loại bỏ trùng lặp <a id="section-III"></a>

**Mục tiêu:** Đảm bảo không có bản ghi nào bị nhân đôi trong các bảng quan trọng. Dữ liệu trùng lặp có thể làm lệch kết quả thống kê và mô hình học máy.

**Các bảng cần kiểm tra:** `orders`, `order_products__prior`, `order_products__train`, `products`

**Kỹ thuật sử dụng:** `df.duplicated().sum()`, `df.drop_duplicates()`




In [13]:
#Kiểm tra giá trị trùng lặp ở các bảng bằng lặp
for name,data_fame in df_all[:4]:
    print(f"\n=== {name} ===")
    print(f'number_duplicated: {data_fame.duplicated().sum()}')



=== order ===
number_duplicated: 0

=== order_products__prior ===
number_duplicated: 0

=== order_products__train ===
number_duplicated: 0

=== products ===
number_duplicated: 0


---
> **Nhận xét:**
>
> Sau khi kiểm tra toàn bộ các bảng, không phát hiện bản ghi trùng lặp nào. Dữ liệu đã sạch và sẵn sàng cho các bước tiếp theo.

---
## IV. Chuẩn hóa kiểu dữ liệu <a id="section-IV"></a>

**Mục tiêu:** Đảm bảo mỗi cột có kiểu dữ liệu phù hợp với ý nghĩa thực tế của nó. Việc này giúp:
- Đảm bảo tính nhất quán khi xuất sang SQL

**Các điều chỉnh cần thực hiện:**

| Cột | Bảng | Kiểu hiện tại | Kiểu mục tiêu | Lý do |
|---|---|---|---|---|
| `product_name` | `products` | `object` | `str` (xóa dấu phẩy) | Tránh R đọc nhầm thành nhiều sản phẩm khi chạy Apriori |

**Kỹ thuật sử dụng:** `df[col].astype()`

In [14]:
# Xóa dấu phẩy trong tên sản phẩm để tránh lỗi khi xuất file transactions cho R
df_pr['product_name'] = df_pr['product_name'].str.replace(',', '_', regex=False)

In [15]:
# Lệnh kiểm tra lại tên sản phẩm
df_pr['product_name'].sample(10)

32788                                Soothing Vapor Bath
34303                          Organic Red Lentil Rotini
17574              Babies Liquid Multivitamin Supplement
43211             Chocolate Reese's Frozen Dairy Dessert
38375                             Clean Energy Zero Lime
36909                                  Crispy Sauerkraut
47719                                        Honey_ Pure
12473                                Chai Green Tea Bags
6898     Boneless Skinless Chicken Breasts with Rib Meat
20878                  Granola_ Peanut Butter_ Homestyle
Name: product_name, dtype: str


---
> **Nhận xét:**
>
> Sau khi chuẩn hóa, phát hiện **3,220 sản phẩm** trong cột `product_name` có tên chứa dấu phẩy — các dấu phẩy này đã được xóa để tránh R đọc nhầm thành nhiều sản phẩm khác nhau khi chạy thuật toán Apriori trong R.

---
## V. Nạp dữ liệu vào SQL Server <a id="section-V"></a>

**Mục tiêu:** Nạp toàn bộ dữ liệu đã làm sạch vào SQL Server, database `Project_ADY201m`. Thành viên phụ trách SQL sẽ dùng database này để:
- Tính chỉ số RFM (Recency, Frequency, Monetary)
- Phân tích hành vi theo thời gian
- Tạo `feature_table.csv` làm đầu vào cho mô hình học máy

**Các bảng được nạp vào cơ sở dữ liệu:**

| Tên bảng | Nguồn | Ghi chú |
|---|---|---|
| `orders` | `df_o` | Đã có cột `is_first_order` |
| `order_products_prior` | `df_opp` | Dữ liệu lịch sử đơn hàng |
| `order_products_train` | `df_opt` | Tập huấn luyện |
| `products` | `df_p` | Thông tin sản phẩm |
| `aisles` | `df_a` | Danh mục |
| `departments` | `df_d` | Nhóm hàng |

**Quy trình thực hiện:**
1. Kết nối tới `master`, xóa database `Project_ADY201m` cũ (nếu có) và tạo mới hoàn toàn — đảm bảo mỗi lần chạy đều là bảng sạch, không còn khóa ngoại của lần chạy trước xen lẫn
2. Định nghĩa cấu trúc các bảng thông qua `sqlalchemy.orm` và `declarative_base()`
3. Nạp dữ liệu từ từng DataFrame vào bảng tương ứng bằng `df.to_sql()`

**Kỹ thuật sử dụng:** `sqlalchemy.create_engine()`, `declarative_base()`, `df.to_sql()` với tham số `if_exists='append'`



In [16]:
# Lấy các công cụ cần dùng trong thư viện sqlalchemy để kết nối với SQL
from sqlalchemy import create_engine, text

# 1. Khai báo các biến chứa thông tin cấu hình
SERVER   = '.'
DATABASE = 'Project_ADY201m'
USERNAME = 'ADY201m-Project'
PASSWORD = '123'

# 2. Kết nối tới 'master' để có quyền DROP/CREATE DATABASE
#    (không thể xóa một database trong khi đang kết nối chính vào nó)
admin_url = (
    f"mssql+pyodbc://{USERNAME}:{PASSWORD}@{SERVER}/master"
    f"?driver=ODBC+Driver+17+for+SQL+Server&Encrypt=no"
)
admin_engine = create_engine(admin_url, fast_executemany=True)

# 3. Xóa database cũ (nếu có) rồi tạo mới hoàn toàn — đảm bảo mỗi lần chạy
#    đều là bảng sạch 100%, không còn tình trạng nửa cũ nửa mới khiến
#    khóa ngoại bị thiếu khi vẽ diagram trong SSMS
try:
    with admin_engine.connect().execution_options(isolation_level='AUTOCOMMIT') as connection:
        connection.execute(text(f"""
            IF DB_ID('{DATABASE}') IS NOT NULL
            BEGIN
                ALTER DATABASE [{DATABASE}] SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
                DROP DATABASE [{DATABASE}];
            END
        """))
        connection.execute(text(f"CREATE DATABASE [{DATABASE}]"))
    print(f"Đã tạo mới Database '{DATABASE}' — sạch hoàn toàn, không còn bảng/khóa của lần chạy trước.")
except Exception as e:
    print(f"Đã xảy ra lỗi khi tạo database: {e}")
    raise

admin_engine.dispose()

# 4. Kết nối chính thức tới database vừa tạo cho các bước tiếp theo
connection_url = (
    f"mssql+pyodbc://{USERNAME}:{PASSWORD}@{SERVER}/{DATABASE}"
    f"?driver=ODBC+Driver+17+for+SQL+Server"
    f"&Encrypt=no"
)
engine = create_engine(
    connection_url,
    fast_executemany=True
)

Đã tạo mới Database 'Project_ADY201m' — sạch hoàn toàn, không còn bảng/khóa của lần chạy trước.


In [17]:
from sqlalchemy.orm import declarative_base
from sqlalchemy import Column, Integer, String, Float, ForeignKey

Base = declarative_base()

# Tạo bảng orders
class Orders(Base):
    __tablename__ = 'orders'
    order_id                = Column(Integer, primary_key=True, autoincrement=False)
    user_id                 = Column(Integer)
    eval_set                = Column(String)
    order_number            = Column(Integer)
    order_dow               = Column(Integer)
    order_hour_of_day       = Column(Integer)
    days_since_prior_order  = Column(Float)
    is_first_order          = Column(Integer)

# Tạo bảng order_products_prior
class OrderProductsPrior(Base):
    __tablename__ = 'order_products_prior'
    order_id            = Column(Integer, ForeignKey('orders.order_id'),     primary_key=True, autoincrement=False)
    product_id          = Column(Integer, ForeignKey('products.product_id'), primary_key=True, autoincrement=False)
    add_to_cart_order   = Column(Integer)
    reordered           = Column(Integer)

# Tạo bảng order_products_train
class OrderProductsTrain(Base):
    __tablename__ = 'order_products_train'
    order_id            = Column(Integer, ForeignKey('orders.order_id'),     primary_key=True, autoincrement=False)
    product_id          = Column(Integer, ForeignKey('products.product_id'), primary_key=True, autoincrement=False)
    add_to_cart_order   = Column(Integer)
    reordered           = Column(Integer)

# Tạo bảng products
class Products(Base):
    __tablename__ = 'products'
    product_id      = Column(Integer, primary_key=True, autoincrement=False)
    product_name    = Column(String)
    aisle_id        = Column(Integer, ForeignKey('aisles.aisle_id'))
    department_id   = Column(Integer, ForeignKey('departments.department_id'))

# Tạo bảng aisles
class Aisles(Base):
    __tablename__ = 'aisles'
    aisle_id    = Column(Integer, primary_key=True, autoincrement=False)
    aisle       = Column(String)

# Tạo bảng departments
class Departments(Base):
    __tablename__ = 'departments'
    department_id   = Column(Integer, primary_key=True, autoincrement=False)
    department      = Column(String)

print("Đã khai báo xong cấu trúc bảng.")

Đã khai báo xong cấu trúc bảng.


In [18]:
# Database vừa được tạo mới hoàn toàn ở bước trên nên không còn bảng cũ —
# chỉ cần create_all(), không cần drop_all() nữa
Base.metadata.create_all(engine)
print("Đã tạo xong toàn bộ bảng trong database, đầy đủ khóa ngoại")

Đã tạo xong toàn bộ bảng trong database, đầy đủ khóa ngoại


In [19]:

MAX_PARAMS = 2000 

# Danh sách bảng theo đúng thứ tự Foreign Key kèm theo chỉ số Ch
# (aisles, departments trước → products → orders → order_products)
df_load_all = [
    (df_a,   'aisles',                 MAX_PARAMS // len(df_a.columns)),
    (df_de,  'departments',            MAX_PARAMS // len(df_de.columns)),
    (df_pr,  'products',               MAX_PARAMS // len(df_pr.columns)),
    (df_o,   'orders',                 MAX_PARAMS // len(df_o.columns)),
    (df_opp, 'order_products_prior',   MAX_PARAMS // len(df_opp.columns)),
    (df_opt, 'order_products_train',   MAX_PARAMS // len(df_opt.columns)),
]

# Dùng lặp để nạp dữ liệu vào sql
for df, table_name, chunk in df_load_all:
    df.to_sql(table_name, engine, if_exists='append', index=False, chunksize=chunk, method='multi')
    print(f"Đã nạp {table_name}: {len(df):,} dòng")

print("Hoàn tất nạp dữ liệu vào database")

Đã nạp aisles: 134 dòng
Đã nạp departments: 21 dòng
Đã nạp products: 49,688 dòng
Đã nạp orders: 3,421,083 dòng
Đã nạp order_products_prior: 32,434,489 dòng
Đã nạp order_products_train: 1,384,617 dòng
Hoàn tất nạp dữ liệu vào database


---

> **Nhận xét:**
>
> Toàn bộ 6 bảng đã được nạp thành công vào database với tổng cộng **37,290,032 dòng** dữ liệu. Quá trình nạp mất khoảng **100 phút** do khối lượng dữ liệu rất lớn, đặc biệt bảng `order_products_prior` chiếm phần lớn thời gian với hơn 32 triệu dòng.
>
> | Bảng | Số dòng |
> |---|---|
> | `aisles` | 134 |
> | `departments` | 21 |
> | `products` | 49,688 |
> | `orders` | 3,421,083 |
> | `order_products_prior` | 32,434,489 |
> | `order_products_train` | 1,384,617 |
>


---
##  Tổng kết Python Data Processing

Python Data Processing đã hoàn thành các bước làm sạch và chuẩn bị dữ liệu. Bảng dưới tóm tắt những gì đã được thực hiện:

| Bước | Nội dung | Kết quả |
|---|---|---|
| I | Đọc và kiểm tra tổng quan 6 bảng |  Nắm được cấu trúc dữ liệu |
| II | Xử lý missing values |  Điền 0 cho đơn hàng đầu tiên, thêm cột `is_first_order` |
| III | Loại bỏ trùng lặp |  Dữ liệu sạch, không có bản ghi nhân đôi |
| IV | Chuẩn hóa kiểu dữ liệu |  Các cột có kiểu dữ liệu phù hợp |
| V | Nạp dữ liệu vào SQL Server |  6 bảng, tổng 37,290,032 dòng, database `Project_ADY201m` |

**Dữ liệu đã sẵn sàng cho các bước tiếp theo:**
- Database `Project_ADY201m` trên SQL Server (được tạo mới hoàn toàn mỗi lần chạy) → thành viên phụ trách SQL tiếp tục feature engineering, xuất `transactions.csv` cho R
- Cùng database này được Python EDA đọc trực tiếp qua SQLAlchemy

---
*Notebook tiếp theo: **Python EDA — Khám phá và trực quan hóa dữ liệu***